# 02 Define Precise Pixel Mask

            Loads the HDF5 data dictionary from notebook 01, lets you draw or edit
            the precise `mask_pixel`, and writes it back into the same HDF5 file.
            If this notebook is skipped, notebook 01 already saved `mask_pixel` as
            an array of zeros.

In [ ]:
import os, sys
from os.path import join
from getpass import getuser

import h5py
import numpy as np
import matplotlib.pyplot as plt
import skimage.morphology
from pyFAI.detectors import Detector

def find_basefolder(start=None):
    folder = os.path.abspath(start or os.getcwd())
    for _ in range(8):
        if os.path.isdir(join(folder, "library")) and os.path.isdir(join(folder, "raw")):
            return folder
        parent = os.path.dirname(folder)
        if parent == folder:
            break
        folder = parent
    raise FileNotFoundError("Could not find beamtime root containing library/ and raw/.")

BASEFOLDER = find_basefolder()
sys.path.append(join(BASEFOLDER, "library"))
print("Beamtime root:", BASEFOLDER)
import fthcore as fth
import helper_functions as helper
import interactive
from interactive import cimshow
import mask_lib
import reconstruct_rb as rec
import PETRA_MaxP04_loading as loading
import fth_phase_workflow as wf

try:
    import cupy as cp
    import cupyx as cpx
    import CCI_core_cupy as cci
    import Phase_Retrieval as PhR
    GPU = True
    print("GPU available")
except Exception:
    import CCI_core as cci
    PhR = None
    GPU = False
    print("GPU unavailable")

%matplotlib widget
try:
    %load_ext jupyter_black
except Exception:
    pass

In [ ]:
BASEFOLDER = find_basefolder()
DATA_H5 = join(BASEFOLDER, "processed", "Logs", "data_recon_ImId_1269_rb.hdf5")
data = wf.load_data_dict(DATA_H5)
positive_label = data["positive_label"]
reference_label = data["reference_label"]
raw_shape = np.asarray(data["holo"][positive_label]["image"]).shape
mask_pixel = np.asarray(data.get("mask_pixel", np.zeros(raw_shape)), dtype=float)
polygon_coordinates = data.get("polygon_coordinates", [])
print("Loaded:", DATA_H5)
print("Raw mask shape:", mask_pixel.shape)

## Paint mask with PNG

In [ ]:
# Option: export a log-scaled detector image, paint mask pixels bright red,
# save the edited image as *_1.png, then run the next cell.
mask_png, mask_painted_png = wf.mask_png_paths(
    BASEFOLDER,
    "mask_pixel",
    data["holo"][positive_label]["id"],
)
wf.save_mask_reference_png(
    data["holo"][positive_label]["image"],
    mask_png,
    log_scale=True,
)
print("Paint bright red mask pixels in:")
print(mask_png)
print("Save the edited PNG as:")
print(mask_painted_png)

In [ ]:
# Option: load bright-red pixels from the edited PNG as mask_pixel.
mask_pixel = wf.load_bright_red_mask_png(
    mask_painted_png,
    expected_shape=mask_pixel.shape,
)
polygon_coordinates = []

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(data["holo"][positive_label]["image"], cmap="gray")
ax[0].imshow(mask_pixel, alpha=0.35, cmap="Reds")
ax[0].set_title("painted mask_pixel overlay")
ax[1].imshow(mask_pixel, cmap="gray")
ax[1].set_title("painted mask_pixel")

## Draw polygon masks

In [ ]:
# Draw polygons on the raw detector image. Click "Add mask" for each polygon,
# then execute the next cell.
plt.close("all")
mask_image = np.ascontiguousarray(
    np.asarray(data["holo"][positive_label]["image"], dtype=np.float32)
)
mask_drawer = interactive.draw_polygon_mask(
    mask_image,
    display_bin=2,
)

In [ ]:
# Option A: from the polygon widget above.
polygon_coordinates = mask_drawer.get_vertice_coordinates()

# Option B: fallback, type coordinates manually before running this cell:
# polygon_coordinates = [[(y0, x0), (y1, x1), (y2, x2)]]

if len(polygon_coordinates) > 0:
    mask_pixel = mask_lib.create_polygon_mask(mask_pixel.shape, polygon_coordinates)
mask_pixel = (np.asarray(mask_pixel) > 0).astype(np.uint8)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(data["holo"][positive_label]["image"], cmap="gray")
ax[0].imshow(mask_pixel, alpha=0.35, cmap="Reds")
ax[0].set_title("raw mask_pixel overlay")
ax[1].imshow(mask_pixel, cmap="gray")
ax[1].set_title("raw mask_pixel")

## Optional threshold additions

In [ ]:
# Optional: add saturated pixels from all raw holograms.
add_oversaturated_pixels = True
oversaturation = data["experimental_setup"].get("oversaturation", np.inf)

if add_oversaturated_pixels:
    for state in data["holo"].values():
        mask_pixel = np.maximum(mask_pixel, np.asarray(state["image"]) > oversaturation)
mask_pixel = (mask_pixel > 0).astype(np.uint8)
cimshow(mask_pixel)

## Center and save mask_pixel

In [ ]:
mask_pixel_png = wf.save_binary_mask_png(
    BASEFOLDER,
    "mask_pixel",
    data["holo"][positive_label]["id"],
    mask_pixel,
)
mask_pixel_c = wf.center_image(mask_pixel, data["center"], cci)
mask_pixel_c = (mask_pixel_c > 0.5).astype(np.uint8)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(data["holo"][positive_label]["image"], cmap="gray")
ax[0].imshow(mask_pixel, alpha=0.35, cmap="Reds")
ax[0].set_title("raw mask_pixel")
ax[1].imshow(data["holo"][positive_label]["image_c"], cmap="gray")
ax[1].imshow(mask_pixel_c, alpha=0.35, cmap="Reds")
ax[1].set_title("centered preview")
for key in [
    "dark_id_im",
    "dark_id_topo",
    "im_id",
    "topo_id",
    "fth_hologram",
    "fth_hologram_unmasked",
    "fth_png_title",
    "fth_recon",
    "fth_recon_unmasked",
    "mask_pixel_smooth",
    "mask_multiplier",
    "sum_c",
    "diff_c",
    "mask_pixel_c",
    "mask_pixel_c_png",
    "prop_dist",
    "phase",
    "dx",
    "dy",
    "focus_operation",
    "roi",
    "recon_cdi",
    "recon_topo_cdi",
    "phase_retrieval_png",
    "roi_cdi",
    "retrieved_type",
    "phase_cdi",
    "prop_dist_cdi",
    "dx_cdi",
    "dy_cdi",
    "focus_mode_cdi",
    "roi_crop",
    "mask_bs_cdi",
]:
    data.pop(key, None)
data.update(
    {
        "mask_pixel": mask_pixel,
        "polygon_coordinates": polygon_coordinates,
        "mask_pixel_png": mask_pixel_png,
    }
)
wf.save_data_dict(data, DATA_H5, overwrite=True)
print("Updated mask_pixel in:", DATA_H5)
print("Saved mask PNG:", mask_pixel_png)

In [ ]:
# Workflow summary
_summary_data = data if "data" in globals() and isinstance(data, dict) else {}
_summary_h5 = globals().get("DATA_H5", _summary_data.get("data_file", "n/a"))
_summary_holo = _summary_data.get("holo", {})
_summary_pos = _summary_data.get("positive_label", globals().get("positive_label", None))
_summary_ref = _summary_data.get("reference_label", globals().get("reference_label", None))
_summary_im = _summary_holo.get(_summary_pos, {}).get("id", globals().get("im_id", "n/a"))
_summary_topo = _summary_holo.get(_summary_ref, {}).get("id", globals().get("topo_id", "n/a"))
print("im_id:", _summary_im)
print("topo_id:", _summary_topo)
print("HDF5:", _summary_h5)